In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files

# sklearn
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

# ??????? ??? ???????? ????? ???????
def manual_upload(file_name):
    print(f"???? {file_name} ?? ??????. ??????????, ????????? ???? ???????.")
    uploaded = files.upload()
    if file_name in uploaded:
        print(f"???? {file_name} ??????? ????????.")
        return file_name
    else:
        raise FileNotFoundError(f"???? {file_name} ?? ??? ????????.")


In [ ]:
# ???? ? ?????????
CLASSIFICATION_DATA_PATH = "diabetes_data_upload.csv"
REGRESSION_DATA_PATH = "laptop_price_data.csv"

# ????????? ??????? ??? ????????????? (????? ????????? ???????)
if os.path.exists(CLASSIFICATION_DATA_PATH):
    df_clf = pd.read_csv(CLASSIFICATION_DATA_PATH)
else:
    CLASSIFICATION_DATA_PATH = manual_upload("diabetes_data_upload.csv")
    df_clf = pd.read_csv(CLASSIFICATION_DATA_PATH)

print("????? ???????? (?????????????):", df_clf.shape)
print(df_clf.head())

# ????????? ??????? ??? ????????? (???? ?????????)
if os.path.exists(REGRESSION_DATA_PATH):
    df_reg = pd.read_csv(REGRESSION_DATA_PATH)
else:
    REGRESSION_DATA_PATH = manual_upload("laptop_price_data.csv")
    df_reg = pd.read_csv(REGRESSION_DATA_PATH)

print("\n????? ???????? (?????????):", df_reg.shape)
print(df_reg.head())

# ???????? ??????? ????????/??????? ?????? (????????? EDA ?? ???1)
print("\n?????????? ? ???????? ?????????????:")
print(df_clf.info())
print("\n?????????? ? ???????? ?????????:")
print(df_reg.info())


????? 2 ? ???????? ????????? (Decision Tree)

In [ ]:
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# -------------------------
# ????????????? (Decision Tree)
# -------------------------

target_col_clf = 'class'
X_clf = df_clf.drop(columns=[target_col_clf])
y_clf = df_clf[target_col_clf]

# 1) ???????? ??? ?????????????? ???????? ? X_clf
categorical_columns = X_clf.select_dtypes(include=['object']).columns
for col in categorical_columns:
    le = LabelEncoder()
    X_clf[col] = le.fit_transform(X_clf[col])

# 2) ???? y_clf ???? ????????? (object), ???????? ???
if y_clf.dtype == 'object':
    le_y = LabelEncoder()
    y_clf = le_y.fit_transform(y_clf)

# ????????? ?? train/test
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42
)

# ??????? ?????? (????????)
dtc_base = DecisionTreeClassifier(random_state=42)
dtc_base.fit(X_train_c, y_train_c)

# ???????????? ? ???????
y_pred_c_base = dtc_base.predict(X_test_c)
acc_base = accuracy_score(y_test_c, y_pred_c_base)
print("=== ???????? (DecisionTreeClassifier) ===")
print("Accuracy:", acc_base)
print("Confusion Matrix:")
print(confusion_matrix(y_test_c, y_pred_c_base))
print("Classification Report:")
print(classification_report(y_test_c, y_pred_c_base))

# ????????????: Confusion Matrix (heatmap)
plt.figure(figsize=(5,4))
sns.heatmap(confusion_matrix(y_test_c, y_pred_c_base), annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix (Baseline Decision Tree)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

# -------------------------
# ????????? (Decision Tree)
# -------------------------
# ??????: target_col_reg = 'Price'
target_col_reg = 'Price'
X_reg = df_reg.drop(columns=[target_col_reg])
y_reg = df_reg[target_col_reg]

# ???? ? ?????? ????????? ???? ???? ?????????????? ????????, ?????????? ??:
cat_cols_reg = X_reg.select_dtypes(include=['object']).columns
for col in cat_cols_reg:
    le = LabelEncoder()
    X_reg[col] = le.fit_transform(X_reg[col])

# ?????????
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

# ??????? ?????? (????????)
dtr_base = DecisionTreeRegressor(random_state=42)
dtr_base.fit(X_train_r, y_train_r)

# ???????????? ? ???????
y_pred_r_base = dtr_base.predict(X_test_r)
mse_base = mean_squared_error(y_test_r, y_pred_r_base)
r2_base = r2_score(y_test_r, y_pred_r_base)
print("\n=== ???????? (DecisionTreeRegressor) ===")
print("MSE:", mse_base)
print("R^2:", r2_base)

# ????????????: scatter "y_test vs y_pred"
plt.figure(figsize=(6,6))
plt.scatter(y_test_r, y_pred_r_base, color='blue', alpha=0.6)
plt.plot([y_test_r.min(), y_test_r.max()],
         [y_test_r.min(), y_test_r.max()],
         color='red', linestyle='--')
plt.title("Baseline Decision Tree: Real vs. Predicted")
plt.xlabel("Real")
plt.ylabel("Predicted")
plt.show()


????? 3 ? ????????? ?????????

3.a ? ???????????? ???????? (?????????????, ????? ????????, ?????? ???????????????)

??? ???????? ??????? ?????? ???????? ??????????????? ????? GridSearchCV:
max_depth, min_samples_split, min_samples_leaf ? ?.?.

In [ ]:
# -------------------------
# ?????????????
# -------------------------
param_grid_clf = {
    'max_depth': [3, 5, 7, 10, None],
    'min_samples_leaf': [1, 2, 5, 10]
}

grid_search_clf = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid_clf,
    cv=3,
    n_jobs=-1
)
grid_search_clf.fit(X_train_c, y_train_c)

print("?????? ????????? (DecisionTreeClassifier):", grid_search_clf.best_params_)
print("?????? ???? ?? CV:", grid_search_clf.best_score_)

# ??????? ?????? ? ??????? ???????????
dtc_improved = DecisionTreeClassifier(
    random_state=42,
    **grid_search_clf.best_params_
)
dtc_improved.fit(X_train_c, y_train_c)

y_pred_c_improved = dtc_improved.predict(X_test_c)
acc_improved = accuracy_score(y_test_c, y_pred_c_improved)
print("\n=== ?????????? ?????? (DecisionTreeClassifier) ===")
print("Accuracy:", acc_improved)
print("Classification Report:")
print(classification_report(y_test_c, y_pred_c_improved))

# ?????????
print(f"\n????????? (Accuracy) ? ????????: {acc_base:.3f} vs ??????????: {acc_improved:.3f}")


# -------------------------
# ?????????
# -------------------------
param_grid_reg = {
    'max_depth': [3, 5, 7, 10, None],
    'min_samples_leaf': [1, 2, 5, 10]
}

grid_search_reg = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    param_grid_reg,
    cv=3,
    n_jobs=-1
)
grid_search_reg.fit(X_train_r, y_train_r)

print("\n?????? ????????? (DecisionTreeRegressor):", grid_search_reg.best_params_)
print("?????? ???? ?? CV (R^2):", grid_search_reg.best_score_)

dtr_improved = DecisionTreeRegressor(
    random_state=42,
    **grid_search_reg.best_params_
)
dtr_improved.fit(X_train_r, y_train_r)

y_pred_r_improved = dtr_improved.predict(X_test_r)
mse_improved = mean_squared_error(y_test_r, y_pred_r_improved)
r2_improved = r2_score(y_test_r, y_pred_r_improved)
print("\n=== ?????????? ?????? (DecisionTreeRegressor) ===")
print("MSE:", mse_improved)
print("R^2:", r2_improved)
print(f"\n????????? (MSE) ? ????????: {mse_base:.3f} vs ??????????: {mse_improved:.3f}")
print(f"????????? (R^2)  ? ????????: {r2_base:.3f} vs ??????????: {r2_improved:.3f}")


# scatter "y_test vs y_pred"
plt.figure(figsize=(6,6))
plt.scatter(y_test_r, y_pred_r_improved, color='green', alpha=0.6)
plt.plot([y_test_r.min(), y_test_r.max()],
         [y_test_r.min(), y_test_r.max()],
         color='red', linestyle='--')
plt.title("Improved Decision Tree: Real vs. Predicted")
plt.xlabel("Real Price")
plt.ylabel("Predicted Price")
plt.show()

# ?????: ?????, ??? ???????? (accuracy, R^2, etc.) ?????????? ????? ??????? ???????????????.


????? 4 ? ????????????? ?????? ??????

????? ????? ?????????? ?????????? Decision Tree (??? ????????????? ?/??? ?????????).

In [ ]:
class MyDecisionStumpClassifier:
    def __init__(self):
        self.feature_index = None
        self.threshold = None
        self.pred_left = None
        self.pred_right = None

    def fit(self, X, y):
        """
        ??????? ????? ??????? ? ?????, ????? ?????????????? ?????? (gini/entropy).
        ??? ????????? - brute force: ?????????? ??? ???????? ? ??? ?????????? ????????
        """
        best_accuracy = -1
        for feature_i in range(X.shape[1]):
            values = np.unique(X.iloc[:, feature_i])
            for val in values:
                # ????????? ?? ?????/?????? ?????
                left_mask = X.iloc[:, feature_i] <= val
                right_mask = ~left_mask
                if len(y[left_mask]) == 0 or len(y[right_mask]) == 0:
                    continue
                left_pred = np.bincount(y[left_mask]).argmax()
                right_pred = np.bincount(y[right_mask]).argmax()

                # ?????????????
                y_pred = np.array([left_pred if m else right_pred for m in left_mask])

                # ??????? accuracy
                acc = accuracy_score(y, y_pred)
                if acc > best_accuracy:
                    best_accuracy = acc
                    self.feature_index = feature_i
                    self.threshold = val
                    self.pred_left = left_pred
                    self.pred_right = right_pred

    def predict(self, X):
        left_mask = X.iloc[:, self.feature_index] <= self.threshold
        return np.array([
            self.pred_left if m else self.pred_right
            for m in left_mask
        ])

# -------------------------
# ?????? ? ???????
# -------------------------
my_dsc = MyDecisionStumpClassifier()
my_dsc.fit(X_train_c, y_train_c)
y_pred_my = my_dsc.predict(X_test_c)
acc_my = accuracy_score(y_test_c, y_pred_my)
print("=== ?????????? ?????? (DecisionStumpClassifier) ===")
print("Accuracy:", acc_my)
print("Confusion Matrix:")
print(confusion_matrix(y_test_c, y_pred_my))
print("Classification Report:")
print(classification_report(y_test_c, y_pred_my))

print(f"????????? ? ?????????? (Accuracy) ? ????????: {acc_base:.3f} vs ??????????: {acc_my:.3f}")

# ???????????? confusion matrix
plt.figure(figsize=(5,4))
sns.heatmap(confusion_matrix(y_test_c, y_pred_my), annot=True, fmt='d', cmap='Purples')
plt.title("Confusion Matrix (MyDecisionStumpClassifier)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()
